<a href="https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/03_vision_foundation_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Maritime Vision Applications
## Session 1 · Part 2 — Vision Foundation Models: the CNN era
*Torben Globisch · VACOT*

---

| Notebook | Topic |
|---|---|
| 00 | Artificial Neural Networks: neurons, layers, learning |
| 01 | Computer Vision & an MLP on MNIST |
| 02 | Convolutional Neural Networks |
| **→ 03** | **Vision Foundation Models: the CNN era** |
| 04 | Vision Transformers |
| 05 | Self-supervised pretraining & transfer |

**Session goal.** Starting from the convolution operation you already know, we build up to the two dominant families
of Vision Foundation Models: **CNN-based** (ResNet, EfficientNet, …, *this notebook*) and **Transformer-based** (ViT, DINOv2, …, *notebooks 04 & 05*).

**By the end of this notebook you can**
1. define a Vision Foundation Model (VFM), explain why the term is deliberately **vague**, and describe its three-phase lifecycle,
2. explain why **ImageNet** changed everything, and how CNN architectures evolved from AlexNet to ResNet and EfficientNet,
3. use a pretrained ResNet as a **feature extractor** and evaluate it with a **linear probe**.

> 💡 Use a GPU runtime: *Runtime → Change runtime type → T4 GPU*.
> This is a **show-and-tell** notebook for the lecture: run it top to bottom, there are no exercises. 🎤 marks prompts for the presenter: questions for the audience (click to reveal the answer) or a value to change live.

In [ ]:
# @title ⚙️ Setup: imports, colours & helpers — run me first { display-mode: "form" }
import math, io, re, base64, time, urllib.request
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import HTML, display
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colour palette — the same colours as the lecture slides
NAVY, BLUE, LAVENDER, ORANGE, GREEN, RED, GREY, LIGHT = (
    "#2D3A8C", "#5B67B8", "#A3A9DB", "#E8702A", "#3E9E64", "#C0392B", "#8A8FA8", "#EEF0F7")
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": NAVY,
    "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, GREEN, BLUE, RED, GREY]),
    "animation.html": "jshtml", "animation.embed_limit": 80,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU")

def play(anim):
    """Show a matplotlib animation as an HTML player (play / pause / step frame by frame)."""
    plt.close(anim._fig)
    def shrink(m):  # re-encode every frame as a 256-colour PNG: 3-4x smaller, looks the same
        img = Image.open(io.BytesIO(base64.b64decode(m.group(1).replace("\\\n", ""))))
        buf = io.BytesIO(); img.convert("RGB").quantize(256).save(buf, "png", optimize=True)
        return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
    return HTML(re.sub(r"data:image/png;base64,([A-Za-z0-9+/=\\\n]+)", shrink, anim.to_jshtml()))

# Maritime example photos from the COCO validation set (Flickr, Creative Commons licences).
MARITIME = {"trawler": 330369, "sailboat": 286422, "lifeboat": 400803, "marina": 442822,
            "fishing_boat": 451090, "yacht": 489014, "ferry": 543300}

def load_image(name, size=None):
    """Load one of the MARITIME photos (or any image URL) as an RGB PIL image."""
    url = f"http://images.cocodataset.org/val2017/{MARITIME[name]:012d}.jpg" if name in MARITIME else name
    with urllib.request.urlopen(url, timeout=30) as r:
        img = Image.open(io.BytesIO(r.read())).convert("RGB")
    if size is not None:  # centre-crop to a square, then resize
        s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC)
    return img

---
# 1 · From learned kernels to foundation models

Notebook 02 showed that kernels are **learned**, not hand-crafted: from a random start, gradient descent found the Sobel
kernel by itself, and a CNN trained only on class labels grew its own edge detectors (the ResNet-18 kernels at the end).

This notebook takes that idea to its extreme: learn from **millions** of images once, and reuse what was learned everywhere.
That is what a *foundation model* is.

---
# 2 · What is a Vision Foundation Model?

> **Working definition.** A Vision Foundation Model (VFM) is a neural network that
> 1. is **pretrained** on a **large dataset** using a **pretext task**,
> 2. learns **general visual representations**,
> 3. can be used as a starting point for **many downstream tasks** with minimal adaptation.

Pretext task examples: classify the 1000 ImageNet categories (*supervised*), predict missing image patches (*MAE*),
match different crops of the same image (*DINO*).

### ⚠️ "Foundation model" is a deliberately vague term
The name was coined in 2021 (Bommasani et al., Stanford, *"On the Opportunities and Risks of Foundation Models"*) for models
trained on broad data at scale that can be **adapted to a wide range of downstream tasks**. There is **no threshold**: no minimum
number of parameters or images, no required architecture, no required pretext task. A ResNet-50 trained on ImageNet in 2015,
MAE, DINOv2 or CLIP: people call all of them foundation models, with good reason and with some stretching.

What they all must have is one **ability**: their features are **general**, so they can be reused for tasks, classes and
domains they were never trained on, with little or no adaptation. That ability is a matter of degree, so instead of asking
*"is this a foundation model?"*, ask: **"how foundational is it?"** How many tasks does it serve? How much adaptation does it
need? How far from its pretraining data does it still work? We measure exactly this ability in this notebook (linear probe
on vessel types it never saw) and in notebook 05, which closes with a side-by-side comparison of three models.

In [ ]:
# @title 🎨 Figure: the three phases of a foundation model { display-mode: "form" }
fig, ax = plt.subplots(figsize=(15, 4.4)); ax.axis("off"); ax.set_xlim(0, 15); ax.set_ylim(0, 4.6)
phases = [
    ("1. Pretraining", NAVY, ["large dataset", "pretext task", "often no labels"], "ImageNet · LVD-142M\nMAE masking · DINO",
     "The model solves a 'puzzle'\n(predict missing patches,\nmatch crops, classify …)"),
    ("2. Benchmarking", ORANGE, ["frozen backbone", "linear probe / k-NN", "representation quality"], "ImageNet top-1\nADE20k mIoU · COCO AP",
     "How good are the frozen\nfeatures? Separates pretraining\nfrom fine-tuning."),
    ("3. Application", GREEN, ["fine-tune or adapter", "task-specific head", "deploy to real domain"], "maritime detection\nmedical · satellite",
     "Adapt to your task: as simple\nas a linear head on frozen\nfeatures."),
]
for i, (title, col, bullets, ex, note) in enumerate(phases):
    x0 = 0.2 + i * 5
    ax.add_patch(FancyBboxPatch((x0, 1.75), 4.3, 2.7, boxstyle="round,pad=0.05", fc=col, ec=col, alpha=.12))
    ax.add_patch(FancyBboxPatch((x0, 3.85), 4.3, 0.6, boxstyle="round,pad=0.05", fc=col, ec=col))
    ax.text(x0 + 2.15, 4.15, title, ha="center", va="center", color="white", fontweight="bold", fontsize=13)
    ax.text(x0 + 0.2, 3.55, "\n".join("• " + b for b in bullets), va="top", fontsize=10.5, color=NAVY, linespacing=1.5)
    ax.text(x0 + 0.2, 2.35, ex, va="top", fontsize=9, color=col, style="italic")
    ax.text(x0 + 2.15, 1.45, note, ha="center", va="top", fontsize=9.5, color="black")
    if i < 2: ax.annotate("", xy=(x0 + 4.95, 3.1), xytext=(x0 + 4.4, 3.1), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=2))
plt.show()

---
# 3 · ImageNet: the dataset that changed everything

**ImageNet-1k** (ILSVRC, the *ImageNet Large Scale Visual Recognition Challenge*, 2010–2017):
1.28 million training images · 1000 categories · 50,000 validation images.

**2012, the breakthrough:** *AlexNet* (Krizhevsky, Sutskever & Hinton) cut the top-5 error from 26.2 % (2nd place) to
15.3 %. It was the first time a CNN won, and it won by a huge margin. From then on, every winner was a CNN.

In [ ]:
ilsvrc = [(2010, 28.2, "NEC-UIUC", False), (2011, 25.8, "XRCE", False), (2012, 15.3, "AlexNet", True), (2013, 11.7, "ZFNet / Clarifai", True),
          (2014, 6.7, "GoogLeNet", True), (2015, 3.57, "ResNet", True), (2016, 2.99, "Trimps-Soushen", True), (2017, 2.25, "SENet", True)]
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar([str(y) for y, *_ in ilsvrc], [e for _, e, *_ in ilsvrc], color=[NAVY if cnn else GREY for *_, cnn in ilsvrc])
for b, (_, e, name, _) in zip(bars, ilsvrc):
    ax.text(b.get_x() + b.get_width() / 2, e + 0.6, f"{e}%\n{name}", ha="center", fontsize=8.5)
ax.axhline(5.1, color=ORANGE, ls="--", lw=2); ax.text(7.45, 9.0, "human ≈ 5.1 % (dashed)", color=ORANGE, ha="right", fontweight="bold")
ax.set(ylabel="top-5 error (%)", title="ILSVRC winners: top-5 classification error (grey = no CNN)", ylim=(0, 33))
plt.show()

*Top-5 error:* the true class is not among the model's 5 best guesses. The human number comes from Andrej Karpathy
labelling the test set himself in 2014.

## 3.1 · CNN architecture evolution
Each generation introduced **one key architectural idea**. The numbers below come straight from `torchvision`'s
model zoo: accuracy of torchvision's reference training of each architecture (single crop, ImageNet-1k validation top-1).

In [ ]:
from torchvision import models as tvm
zoo = [("AlexNet", 2012, tvm.AlexNet_Weights.IMAGENET1K_V1, "first GPU-trained deep CNN, ReLU, dropout"),
       ("VGG-16", 2014, tvm.VGG16_Weights.IMAGENET1K_V1, "deeper: only 3×3 convolutions"),
       ("GoogLeNet", 2014, tvm.GoogLeNet_Weights.IMAGENET1K_V1, "inception modules, cheap"),
       ("ResNet-50", 2015, tvm.ResNet50_Weights.IMAGENET1K_V1, "skip connections → 100+ layers"),
       ("ResNet-152", 2015, tvm.ResNet152_Weights.IMAGENET1K_V1, ""),
       ("DenseNet-201", 2017, tvm.DenseNet201_Weights.IMAGENET1K_V1, "dense connections"),
       ("EfficientNet-B0", 2019, tvm.EfficientNet_B0_Weights.IMAGENET1K_V1, "compound scaling of depth/width/resolution"),
       ("EfficientNet-B7", 2019, tvm.EfficientNet_B7_Weights.IMAGENET1K_V1, ""),
       ("ConvNeXt-L", 2022, tvm.ConvNeXt_Large_Weights.IMAGENET1K_V1, "a CNN modernised with ViT tricks"),
       ("ViT-B/16", 2021, tvm.ViT_B_16_Weights.IMAGENET1K_V1, "no convolutions: a Transformer (nb. 04)")]
print(f"{'model':<17}{'year':<6}{'params':>9}{'GFLOPs':>8}{'top-1':>8}   key idea")
for name, year, w, idea in zoo:
    m = w.meta
    print(f"{name:<17}{year:<6}{m['num_params'] / 1e6:>8.1f}M{m['_ops']:>8.1f}{m['_metrics']['ImageNet-1K']['acc@1']:>7.1f}%   {idea}")

fig, ax = plt.subplots(figsize=(10, 4.6))
for name, year, w, idea in zoo:
    m = w.meta; acc = m["_metrics"]["ImageNet-1K"]["acc@1"]
    col = ORANGE if "ViT" in name else (GREEN if "ResNet" in name else NAVY)
    ax.scatter(m["_ops"], acc, s=m["num_params"] / 4e5, color=col, alpha=.6, ec="k")
    dx, dy = {"ConvNeXt-L": (-8, 8), "EfficientNet-B7": (8, -12), "DenseNet-201": (6, 4), "ResNet-50": (6, -10)}.get(name, (6, -3))
    ax.annotate(f"{name} ({year})", (m["_ops"], acc), xytext=(dx, dy), textcoords="offset points", fontsize=9, ha="right" if dx < 0 else "left")
ax.set(xscale="log", xlabel="compute per image (GFLOPs, log scale)", ylabel="ImageNet top-1 accuracy (%)",
       title="Accuracy vs. compute (bubble area ∝ number of parameters)")
plt.show()

Bigger is not automatically better: VGG-16 has 138 M parameters, but ResNet-50 is more accurate with 25 M,
and EfficientNet-B0 almost matches it with 5 M. **Architecture ideas matter.**

---
# 4 · The CNN pretraining paradigm
**Train once, reuse everywhere:**

In [ ]:
# @title 🎨 Figure: pretrain once, reuse the backbone everywhere { display-mode: "form" }
fig, ax = plt.subplots(figsize=(14, 4.3)); ax.axis("off"); ax.set_xlim(0, 14); ax.set_ylim(0, 5)
def box(x, y, w, h, text, col, txtcol="white", fs=11):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06", fc=col, ec="none"))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", color=txtcol, fontsize=fs, fontweight="bold")
box(4.5, 4.0, 5, 0.8, "Pretrain on ImageNet\n1.28 M images · 1000 classes · cross-entropy", NAVY, fs=10)
box(5.0, 2.55, 4, 0.8, "Backbone (ResNet-50)\nweights θ*", ORANGE, fs=10)
ax.annotate("", xy=(7, 3.4), xytext=(7, 3.95), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=2))
for i, (task, data) in enumerate([("Classification", "ImageNet labels"), ("Detection", "COCO boxes"), ("Segmentation", "COCO masks"), ("Depth estimation", "depth labels")]):
    x = 0.4 + i * 3.4
    box(x, 0.5, 2.8, 1.0, f"{task}\nfine-tuned on\n{data}", GREEN, fs=9.5)
    ax.annotate("", xy=(x + 1.4, 1.55), xytext=(7, 2.5), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=1.5))
plt.show()

Each downstream model is still fine-tuned on its own labelled data, so these are **task-specific** models built on a shared
backbone. Not yet very "foundational" in the broad sense, but the idea of a reusable pretrained backbone was born.

---
# 5 · Demo: a pretrained ResNet-50 on our own maritime data

**Plan**
1. load a ResNet-50 with ImageNet weights,
2. remove its final classification layer → every image becomes a **2048-dimensional feature vector** (the figure below shows where),
3. explore the feature space of a **maritime dataset** it has never seen: which photos end up close to each other?
4. **linear probe:** freeze the backbone and train only a linear layer on top, then race it against a CNN trained
   **from scratch** on the same (small) set of labels.

*(Note: `pretrained=True` in older tutorials is deprecated; today you pass `weights=...`.)*

In [ ]:
# @title 🎨 Figure: ResNet-50: where the features come from and what we removed { display-mode: "form" }
fig, ax = plt.subplots(figsize=(16, 6.4)); ax.axis("off"); ax.set_xlim(0, 16); ax.set_ylim(0, 6.6)
cy = 3.5
# the frozen backbone sits on a light panel
ax.add_patch(FancyBboxPatch((0.0, 1.0), 11.45, 5.0, boxstyle="round,pad=0.05", fc=LIGHT, ec=LAVENDER, lw=1.5))
ax.text(0.2, 5.75, "frozen ResNet-50 backbone (23.5 M parameters, never updated)", color=NAVY, fontweight="bold", fontsize=12, va="top")

def stack(x, w, h, col, title, sub):
    ax.add_patch(Rectangle((x, cy - h / 2), w, h, fc=col, ec="white", lw=2))
    ax.text(x + w / 2, 1.65, title, ha="center", va="top", fontsize=10, fontweight="bold", color=NAVY)
    ax.text(x + w / 2, 1.2, sub, ha="center", va="top", fontsize=8.5, color=GREY)
def arrow(x0, x1, y0=cy, y1=None, col=NAVY, **kw):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y0 if y1 is None else y1), arrowstyle="-|>", mutation_scale=16, color=col, lw=2, **kw))

# feature maps: height = image size, width = number of channels (like in the CNN figures of notebook 02)
stack(0.25, 0.25, 2.9, "#9AA0B8", "photo", "3 × 224 × 224")
arrow(0.6, 1.2)
stack(1.25, 0.5, 2.5, LAVENDER, "stem", "conv 7×7 + pool\n64 × 56 × 56")
stages = [(2.2, 0.8, 2.0, "layer 1", "3 blocks\n256 × 56 × 56", BLUE), (3.9, 1.1, 1.5, "layer 2", "4 blocks\n512 × 28 × 28", BLUE),
          (5.8, 1.5, 1.0, "layer 3", "6 blocks\n1024 × 14 × 14", BLUE), (8.0, 1.9, 0.6, "layer 4", "3 blocks\n2048 × 7 × 7", NAVY)]
prev = 1.75
for x, w, h, title, sub, col in stages:
    arrow(prev + 0.05, x - 0.05); stack(x, w, h, col, title, sub); prev = x + w
ax.text(5.5, 5.15, "convolutional stages: the image gets smaller (height) and the channels more numerous (width)",
        ha="center", fontsize=9.5, color=BLUE, style="italic")

# global average pooling -> the 2048-d feature vector
arrow(9.95, 10.75)
ax.text(10.35, cy + 0.35, "global\navg pool", ha="center", fontsize=8.5, color=NAVY)
ax.add_patch(Rectangle((10.8, cy - 1.3), 0.4, 2.6, fc=ORANGE, ec="white", lw=2))
for yy in np.linspace(cy - 1.2, cy + 1.2, 9): ax.plot([10.83, 11.17], [yy, yy], color="white", lw=0.8, alpha=.8)
ax.text(11.0, cy + 1.6, "FEATURES  z", ha="center", fontsize=11.5, fontweight="bold", color=ORANGE)
ax.text(11.0, cy - 1.45, "2048 numbers\nper photo", ha="center", va="top", fontsize=9, color=ORANGE, fontweight="bold")

# what we removed (red) and what we add (green)
ax.add_patch(FancyBboxPatch((12.9, 4.5), 2.9, 1.2, boxstyle="round,pad=0.05", fc="white", ec=RED, lw=2.5, ls="--"))
ax.text(14.35, 5.1, "fc layer  2048 → 1000\n(ImageNet classes)", ha="center", va="center", fontsize=10, color=RED, fontweight="bold")
ax.plot([12.95, 15.75], [4.55, 5.65], color=RED, lw=2.5); ax.plot([12.95, 15.75], [5.65, 4.55], color=RED, lw=2.5)
ax.text(14.35, 6.1, "REMOVED  (backbone.fc = nn.Identity())", ha="center", fontsize=10, color=RED, fontweight="bold")
ax.add_patch(FancyBboxPatch((12.9, 1.3), 2.9, 1.2, boxstyle="round,pad=0.05", fc=GREEN, ec="none"))
ax.text(14.35, 1.9, "linear probe  2048 → 6\n(our six vessel types)", ha="center", va="center", fontsize=10, color="white", fontweight="bold")
ax.text(14.35, 0.85, "NEW: the only layer we train (12,294 parameters)", ha="center", fontsize=10, color=GREEN, fontweight="bold")
arrow(11.3, 12.85, cy + 1.0, 5.0, col=RED, linestyle="--")
arrow(11.3, 12.85, cy - 1.0, 1.9, col=GREEN)
plt.tight_layout(); plt.show()

In [ ]:
import torchvision
from torchvision.models import ResNet50_Weights

weights = ResNet50_Weights.IMAGENET1K_V2
backbone = torchvision.models.resnet50(weights=weights)
backbone.fc = nn.Identity()                  # remove the final FC layer (2048 → 1000)
backbone = backbone.eval().to(device)
preprocess = weights.transforms()            # resize 232 → centre-crop 224 → normalise

with torch.no_grad():
    z = backbone(preprocess(load_image("trawler"))[None].to(device))
print("feature vector shape:", tuple(z.shape))

### Our maritime dataset: six vessel types, 50 labelled photos each
450 photos from **Open Images** with human-verified labels: sailboats, speedboats, ferries, cruise ships, cargo/container
ships and fishing vessels. **50 photos per class for training, 25 for testing.** That is a realistic situation for a maritime
project: labelling needs experts and is expensive, so you rarely have more than a few dozen examples per class.
The photos are real-world snapshots (different distances, viewpoints, weather), and some are hard even for humans.

In [ ]:
# @title 🗂️ The maritime dataset: 450 vessel photos from Open Images (run me) { display-mode: "form" }
# Human-verified image-level labels from Open Images V7 (annotations CC BY 4.0, photos from Flickr under CC BY 2.0).
# Photo authors are listed in assets/vessel_photo_credits.csv of the course repository.
import os, urllib.request
from concurrent.futures import ThreadPoolExecutor

VESSEL_IDS = {   # Open Images V7 image ids, 75 per class: the first 50 are the training set, the last 25 the test set
    "sailboat": """
    train/0017a5aca14e8a4f train/270f25208f428a20 train/01b0942287e46f96 train/042a9b5a1326e4bf
    train/4a6e88eca482b411 train/02dc87c4ffccca7e train/2f79b406c757c8ce train/289240aa3b8a1d77
    train/017d6e8b13d944c4 train/02452c3e8af9d4f4 train/0374707afaf5d93c train/15a3fc1ab5932a9f
    train/03725f2f9d997453 train/5cb74411e4e46105 train/0f8ab6b8fdcae79f train/4f601ccdde9eb719
    train/101d084a635efb76 train/000beb11fce5e3f1 train/341fa0c97a6a0c73 train/534223af95e2335b
    train/03d578aa62894dd3 train/02fe072f823a395f train/28846b6efb417e51 train/1d635f9654a4cd46
    train/19c7860b399a67d9 train/078f5ab0461695c8 train/515ff28a9f1214c8 train/09f08dcbc4fc970c
    train/191d9392f9a554ac train/5da8f60b54c25aed train/04a5935005756ce3 train/494267c94fbb7e22
    train/2f44e0335cfb5116 train/097a7d1e539776c2 train/0d2f380aba3fe3ae train/0201e168b1a350d4
    train/4e560f0a0211eab7 train/37955d9033610400 train/016e435568482491 train/19667b9e71c9fde4
    train/17141cd1d4223d10 train/0068ab21287bd575 train/474be7dcb750c0a6 train/2cd21e3ad1b530f4
    train/0112ee98d70904dc train/037303ad5d08ee8f train/30cf161b1f53c069 train/03ecb0ca96badd07
    train/4ca2a417d49b54bf train/01be612e619d1fab train/00268e680a609121 train/1b1eb10e184cadb1
    train/041ce864089465d1 train/010ebc3a9f513bef train/011acb31d18f4e19 train/07409b074d678f45
    train/11322b2db43e5d78 train/27bf2c8347c0756f train/2bdcec9243ec870d train/00a5053c35db851f
    train/114724f782fa1ac0 train/15f6cde66be01c59 train/054a4ee01ad343a4 train/468286170912d314
    train/234b0df2e4415e46 train/09f2edd4abfc4ca4 train/0a2d26bec1379d6d train/55de60f41eba09fe
    train/20045892a78d9d91 train/00d487e5e412be48 train/1f267a7867909f8e train/2bd3b48b53808dac
    train/20552329143c27b9 train/36f5f9763d76921a train/0e9d14c9c6ad80e8""",
    "speedboat": """
    test/7693cd525d384ed9 train/0ff28b7df418c29c train/447afa110efbc0f7 train/02f8c8ac608ae6a8
    train/01a0a526d767aaf1 train/00b934e52ca2240d test/386a81e65f7ba8a3 train/06c14a69d501708c
    train/82c156c2f63ed64a train/25be63ae33cd26eb test/55af54707631f77a train/dceaec9d14811a67
    train/06c2dd40e6ead0aa test/fc17acc08b558526 train/270372e9a68d7a3f train/ae7f44a5d7211736
    test/e311e37f66d84987 train/2bf611ad1fecd61d train/03e9ab6d9e190f18 train/50e0ed72f65c55bc
    test/d7f9c3791f6b3b84 train/0c6b77b7cfffbfcf test/bc1cab21b79505a6 train/a4e9e568527b578a
    train/0e1d00adc201c2ee train/a60d3a3127634a27 test/8877b4b24819da20 test/40d561a18ec315e5 test/c1f39ae6cfe350bd
    train/009135298b681ca3 train/001a0e2feef561c7 train/001a78066f572541 train/02c786e18af7ef0b
    train/085eb22a42f4500e test/55977d5de1b57012 train/3d46c82fdc0b26be train/067314c63fbdccb6
    train/2a36b9f16bf06db6 train/61bf9fa60b0a50ca train/032f4d2a18da0a5b test/f0cfb1215ae2b6e7
    train/3c222ac4a6257007 train/d95b966422e052e1 train/0110b0d005602f45 train/a6b0c488f90b0301
    train/0b25c4f261c87d26 test/72e65794f9f8b127 test/c2c95c571ef57115 train/007741a1be3f3280
    train/068a2062f112a16c train/10699dcd14dfac23 train/21154d871bdf54f4 test/d1d4028c10660c00
    train/308fffd977004718 test/9fe42bf848675308 test/d832f470f2ed2d4c test/bab7df5c8af2d8a3 train/15d98d56fdf37c5c
    train/092babacceaf5bb1 train/7f8f8404369adb38 train/d46ac4c76b9b5596 test/ed3a297b604eec23
    train/06ef034cee641dc9 train/0068af751c04c281 train/06749e42da77b4d3 train/1b50ee203ab8062e
    test/a37975f2d8eb65d2 test/f70169cae0b7e902 train/11b3a8f54fc2520f test/5772bbd05cb7aeba test/c52c840357f49938
    test/bf3201dd617f73cc test/8c18621a9e182de4 train/1240c0d08b65add4 train/016c4230676e2a15""",
    "ferry": """
    test/7a98c371f9d26bf2 test/1f0789232c23b9d2 test/1db6996b94c3af8d train/2020150fa5178ca1 test/6040600c1ec2e472
    test/7af863569ec4a543 train/19d96eee3abe4a84 test/271a63e888b97573 test/ac5baff7c4df4afa train/008be3073e8db77e
    test/f2a4fa73e6042b36 train/005b64d7a7dd7db0 train/3a36c4ecac8edc0c test/08f0e94891f99528 test/f604d0f25e7431a1
    test/7a6c885329d809f0 train/0debff76c6b5ff37 test/1e966f5df38fa8bd test/6b17965d00d4c158 test/b9e728ef283886e0
    test/c4cc47b37532f1b8 test/d2062c74661c9e1a train/49311edf36113265 test/e44496b9f3f3c0d5 train/00818942f783817e
    test/87b8418f60bf6c21 test/ddfce2d828bfb5e7 test/801f919125408088 test/bf0a5586b7b8bc6e test/5b5ffbce8bb67103
    test/5376fee27c87f8eb train/06fe127f335b5bb1 train/0a15968b8a6cb036 train/cbff57622ea2c658
    test/9eec387ad72111a2 train/02c1f4f557ee3d67 test/c36e48ca8d212f37 test/66e87d7c2381f2e0 test/9f4c9587dd89aba0
    test/154e7fac62e5a8d9 train/0204f979aef76229 test/6ca27b11eb161cbc train/13c835164a34991d
    train/00e541d948704a5c test/76725e536b6b9993 train/298c5839727ea7a6 train/268afe6a2d3a7552
    test/e27d79f519db6def test/a6f2076a108797cc test/05cfc55b16cc2b14 train/03c4c0c65e4de86d train/03b774fabee1be48
    train/4f67ac89f5549743 test/0b064592acaae176 train/5815969f9037efe8 train/936dfd8ec3ae0a15
    test/53537dfc83cd5cb4 train/00c9251579624f5d test/75009ae5d08aa0f9 train/009c8b4e2ade1548
    train/47d39754e0af8dd5 test/4d18cf7c46f4ff90 test/87d2058a8a36c914 train/3ac08b77af04dd1f
    train/c91f775749ac1a62 test/b07cf339fdba1b7a test/3ffdc7637447a73a train/01b913fad91691f3
    train/06a18b3568f2efd0 train/175314a3d631192c train/01c2cf0f69ed3b75 train/009bbf88c189af9c
    test/6fd7242ea3968eb3 test/64cf1f42b54d38fd train/b00138ac393a9ba4""",
    "cruise ship": """
    train/1194df37b9507d39 train/21c92b4caf9d650f train/2b7d1b6d8e3a8101 train/0abd34acde73436e
    train/154da29a2b355285 train/6ba557459e405558 train/9443bfdab08c84a5 train/166bb0b95f7f157f
    train/0ceb84244a75f993 train/3d1f93ea04f093a0 train/009bf85434579e4a train/c27f0eec4a63dde9
    train/da2c4aa1639939f8 train/0dfe47147739f9de train/5f4de6a26863b335 train/38c84339d2e7ab2a
    train/19893c7a52d7f065 train/2f2c711a7af0cce1 train/1884bfcce25186ee train/db2b1d1d881f2ab2
    train/0ebd7452b4c1de95 train/10fe56e0ee384e1c train/1e976dbdab3f1f89 train/167d922607de56d6
    train/47599faa7fe8815c train/fbca26e4db84a249 train/00baf4acd29fec99 train/193608f978720f25
    train/14480f52d3a0675d train/0f7567d4a625490c train/603ef2f86f78a33e train/1bb8980e279fe649
    train/87004156745333c6 train/5cea567b01ebce9c train/2695323175048c98 train/108a1677d9109f67
    train/c4bba4aaa5de1edd test/6887b48136d4397e train/06c4a91c722ba57d train/213ceb4d4d329a2e
    train/22217ff40561e30f train/31082c48aa5f0758 train/2b2e19a666a40f60 train/0226208568ff6550
    train/010077f5343afbcf train/104d9780b5021635 train/0181ba314fded795 train/0d999ae353fb5f10
    train/565eab9a3be85b1f train/0f52f4adc7b0d69a train/163c070c3ea395c2 train/4832b041da177924
    train/601f18a94a85734c train/225cdace04e3f927 train/11feb8084d60010c train/13848b2d40d3cf94
    train/306a731865a32388 test/708e654e90b179df train/2b3450813f2c22a5 train/04cd3b06f1c65499
    train/2fa1510cd301d5a8 train/11c97e6a23442521 train/01b3681ee8735786 train/03498b3ce8b7fea4
    train/eec75914faff4f42 train/0c6eb74b168da0c3 train/06b191ded4d6bc18 train/c058d825f61dbdd3
    train/3beafba73da26219 train/0e9364519fc8b490 train/0cc5f9f6c74a8a05 train/1f54da5e6e04c361
    train/fc734066ccb54806 train/3a7e08075667183c train/a40ec067dc2f8df2""",
    "cargo ship": """
    train/1a06ca6e11e53ebb train/6dc973bbd648cd37 test/d6015312d058c5e2 train/40c8e5a6b0c5359a
    train/d2ad679aab8780d6 train/4e912ab82b51efd9 test/627d76aba5ffbffd train/17b2fcaeeb09b271
    train/1631a3f452ab4822 train/07402aa5202e1cc8 test/3d8f90e8fa34060b train/b8d0db4ff71b361c
    train/1ef3cb1d0538e726 train/2c56e263d797e658 train/a62d7d77a34e9968 train/ab92430212fdc31d
    train/2bfdd42f520fe098 test/17ead6e57ffcabec train/1bb4428764418b3c train/131f58acd53530cc
    test/1164019caaac9ac7 train/00b6c0610dcad7d8 train/4ea783ca4357eeed train/01a49b6659c455c3
    train/0a1e2996b16ab13d test/aa23220c2e74390c test/8c1b6c2e0af0df67 train/0cf035cfdde6cbd4
    train/0779a3574a19db16 test/02914fda0e8fcf20 train/031ec9949229c38e train/1804bf398c7b74f0
    train/158646041834ff1b train/151437c63d5d056a train/be74dde43c8e5a09 test/d29139a884c79afa
    train/20fb3051a9455bba train/00bc4549af18ee8c test/c3def997d3332a63 train/00463e9506f3585b
    test/8dd9171a5b1aced2 train/3bce967f1430160b test/fa1b6aac88e3047b test/ecb46f1270bc9b10 train/29cf93f053578bc3
    test/d86fbe30ca29ddb6 train/1a0220308b5cacdd train/0c052c95c2d99bde train/60407e2ca7baa1e5
    train/080a7769c2e7e3a7 train/334dffc8e2c080f6 train/023e3cc3493f6763 train/d2ce0286bc5b3697
    test/a7419eb6679ff41b test/200cf602bb107ea5 train/4aa68d9932e820d1 test/5ad2697b3373b4b8 train/58ae9eb5bf30576d
    train/9b4fa7972b92e489 train/34e91523ebec1c2e train/75e4ad678705b775 train/d333f50ac0443bc5
    train/aebf96cf6280e9e9 train/06438a9a38882a34 train/36e99d49c5e30bc3 test/0b0e79034e64e08a
    train/0e03ad9a75b9934e train/b2c3a870c6faf8c6 test/25efb7b9f68f261b train/2075729770475a5c
    train/640f746ab8ff8721 train/13fc82e48ff64442 train/a12ec0b49f2a0fc0 train/de242501eee6e346
    train/0892f56b5bd4cdb8""",
    "fishing vessel": """
    train/00bffc9f1573d3b8 test/4cbbc0a34c6db938 test/b3782e5235c52733 train/022c67809939cc50
    train/030a01788b3ef318 test/26a88d4e0aff5018 train/00f7e5b6bd9c9f7d train/022a0b2bef00c583
    train/080f5c70438bf53a test/31fe61b00399f0f4 train/01b7c223365aadaf train/031f02e85ae63c7f
    test/1f20eead55d3dec6 train/15d5dc96d6b2f086 train/28de23353fb68135 train/03de0f0b21473f1f
    train/009612f7fc462117 train/0200e4c38e9a191d train/0427eb01462ed45f train/04ae0a59c3e8293f
    train/074ccca9a55a3bfb train/137d396060cdfcae train/00d102ecd20ff393 test/e6fdd541adcede78
    train/1ba7d0be09d14f00 train/00f61144be68b505 train/193ecdd081551a52 train/13d4897d7237294c
    train/0073d756da528f00 train/0339f3e454d4915c train/0034c88824c064a7 train/01092100b8cb5612
    train/1354c27c0b55f873 train/1561f68f83d3df2c train/a5b16ec7734944fc train/75bb854e92675085
    train/0103666f416102e1 train/01f86fd6f882feea train/0a0fc6fae177ead8 train/00b8cdfb794fd452
    train/27ec97a69bb66400 train/28d8c0a9e6b66461 train/07dd8cbd4bc66c99 train/0d69e56109699695
    test/c2679b0f70256712 train/02bca0d3d5b73d77 train/36928ced9ed8e738 train/03b980b6fed5f3e6
    train/095c2c6b32ffa8d1 train/289ee24cd60aa8a1 test/dea0a98898da0a60 train/14c00dd9b83773e9
    train/0d2f3c705bdd93c9 train/00bd0934443df5b2 train/0122a88bf2f1f3a7 train/c486238f81c3994a
    train/1eb83e23c291600a train/08011b1d8efa3a0e train/12e230b3ec3bf57f train/0083cbf8e02aa2f4
    train/10ca246878775d81 train/1b3c94bea20d7ce0 test/dbf5db7cc0fffe7d train/043fb92c60162d51
    train/03612b66016d51b1 train/0292342978c23c13 test/e6c9c7d9d98b1287 test/84f30b598db989f9 test/e6ef32fce751bb90
    train/01138d95e5ec7507 train/030e84ae66194a7d train/06416a56ee5697d8 train/026c1e50529f162a
    train/007154ad137d092f train/1170f52639ded833""",
}
VESSEL_CLASSES = list(VESSEL_IDS)
N_TRAIN_PER_CLASS = 50

def load_vessels(size=224, cache="vessels_224.pt"):
    """Download the photos once (≈ 90 MB), centre-crop them to squares, resize, and cache them as one uint8 tensor."""
    if os.path.exists(cache):
        return torch.load(cache)
    def fetch(key):
        split, image_id = key.split("/")
        for attempt in range(3):
            try:
                with urllib.request.urlopen(f"https://open-images-dataset.s3.amazonaws.com/{split}/{image_id}.jpg", timeout=60) as r:
                    img = Image.open(io.BytesIO(r.read())).convert("RGB")
                s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
                return np.asarray(img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC))
            except Exception:
                time.sleep(1)
        return None                                           # give up on this photo, keep the rest
    items = [(c, i, key) for c, ids in enumerate(VESSEL_IDS.values()) for i, key in enumerate(ids.split())]
    with ThreadPoolExecutor(16) as ex:
        arrays = list(ex.map(lambda item: fetch(item[2]), items))
    keep = [k for k, a in enumerate(arrays) if a is not None]
    data = {"images": torch.tensor(np.stack([arrays[k] for k in keep])).permute(0, 3, 1, 2),     # (N, 3, size, size) uint8
            "labels": torch.tensor([items[k][0] for k in keep]),
            "is_train": torch.tensor([items[k][1] < N_TRAIN_PER_CLASS for k in keep])}
    if len(keep) < len(items): print(f"⚠️  {len(items) - len(keep)} photos could not be downloaded and are skipped")
    torch.save(data, cache)
    return data

In [ ]:
vessels = load_vessels()                                      # ≈ 90 MB download on the first run, then cached
imgs_train, labels_train = vessels["images"][vessels["is_train"]], vessels["labels"][vessels["is_train"]]
imgs_test, labels_test = vessels["images"][~vessels["is_train"]], vessels["labels"][~vessels["is_train"]]
print(f"train: {tuple(imgs_train.shape)}   test: {tuple(imgs_test.shape)}   classes: {VESSEL_CLASSES}")

fig, axes = plt.subplots(len(VESSEL_CLASSES), 8, figsize=(13, 10))
for c, name in enumerate(VESSEL_CLASSES):
    for k, i in enumerate((labels_train == c).nonzero()[:8, 0]):
        axes[c, k].imshow(imgs_train[i].permute(1, 2, 0)); axes[c, k].set_xticks([]); axes[c, k].set_yticks([])
    axes[c, 0].set_ylabel(name, fontsize=11, color=NAVY, rotation=0, ha="right", va="center")
plt.suptitle("Training photos (8 of 50 per class)", color=NAVY, fontweight="bold"); plt.tight_layout(); plt.show()

In [ ]:
@torch.no_grad()
def extract_features(model, imgs_uint8, batch_size=50):
    """Normalise the 224 × 224 photos like ImageNet and run them through the frozen backbone."""
    mean = torch.tensor([0.485, 0.456, 0.406], device=device)[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225], device=device)[:, None, None]
    feats = []
    for i in range(0, len(imgs_uint8), batch_size):
        x = imgs_uint8[i:i + batch_size].to(device).float() / 255
        feats.append(model((x - mean) / std).cpu())
    return torch.cat(feats)

t0 = time.time()
feat_train, feat_test = extract_features(backbone, imgs_train), extract_features(backbone, imgs_test)
print(f"features: train {tuple(feat_train.shape)}, test {tuple(feat_test.shape)}  ({time.time() - t0:.0f}s)")

### Explore the feature space
**t-SNE** squeezes the 2048-D vectors into 2-D while trying to keep neighbours together. The backbone was trained on ImageNet
and has never seen these photos or our six labels, yet photos of the same vessel type already end up close to each other:

In [ ]:
from sklearn.manifold import TSNE
feat_all, labels_all = torch.cat([feat_train, feat_test]), torch.cat([labels_train, labels_test])
emb = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(feat_all.numpy())
VESSEL_COLORS = [BLUE, ORANGE, GREEN, RED, NAVY, "#B07AA1"]
fig, ax = plt.subplots(figsize=(8.5, 6.5))
for c, name in enumerate(VESSEL_CLASSES):
    m = labels_all == c
    ax.scatter(*emb[m].T, s=22, color=VESSEL_COLORS[c], label=name, ec="white", lw=.4, alpha=.9)
ax.legend(markerscale=1.5, bbox_to_anchor=(1, 0.5), loc="center left"); ax.set_xticks([]); ax.set_yticks([])
ax.set_title("t-SNE of frozen ResNet-50 features (all 450 vessel photos)"); plt.tight_layout(); plt.show()

**Image retrieval:** for a typical test photo of each class (the one closest to its class average), find the most similar
training photos (cosine similarity of the features):

In [ ]:
fn = F.normalize(feat_train, dim=1); fq = F.normalize(feat_test, dim=1)
def typical(c):                                       # the test photo of class c closest to the class average
    idx = (labels_test == c).nonzero()[:, 0]
    return idx[(fq[idx] @ F.normalize(fq[idx].mean(0), dim=0)).argmax()].item()
queries = [typical(c) for c in range(len(VESSEL_CLASSES))]
fig, axes = plt.subplots(len(queries), 6, figsize=(11, 11.5))
for r, q in enumerate(queries):
    sims = fn @ fq[q]; nn_idx = sims.topk(5).indices
    axes[r, 0].imshow(imgs_test[q].permute(1, 2, 0)); axes[r, 0].set_title(f"query: {VESSEL_CLASSES[labels_test[q]]}", color=ORANGE, fontsize=9, fontweight="bold")
    for c, j in enumerate(nn_idx):
        axes[r, c + 1].imshow(imgs_train[j].permute(1, 2, 0))
        axes[r, c + 1].set_title(f"{VESSEL_CLASSES[labels_train[j]]} ({sims[j]:.2f})", fontsize=8.5,
                                 color=GREEN if labels_train[j] == labels_test[q] else RED)
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("Nearest neighbours in feature space", color=NAVY, fontweight="bold"); plt.tight_layout(); plt.show()

### Linear probe vs. training from scratch: a race
**Linear probe:** the backbone stays frozen; we only train a single linear layer $\hat y = W z + b$ on the features
($2048 \times 6 + 6 = 12{,}294$ parameters). We record the test accuracy after every epoch:

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
EPOCHS = 40
SHOWCASE = queries                                     # one test photo per class, shown in the race animation

torch.manual_seed(0)
probe = nn.Linear(feat_train.shape[1], len(VESSEL_CLASSES))
opt = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=1e-4)
probe_acc, probe_pred = [], []
for epoch in range(EPOCHS + 1):
    if epoch > 0:                                      # epoch 0 = before any training
        for f, y in DataLoader(TensorDataset(feat_train, labels_train), batch_size=32, shuffle=True):
            loss = F.cross_entropy(probe(f), y)
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        pred = probe(feat_test).argmax(1)
    probe_acc.append((pred == labels_test).float().mean().item()); probe_pred.append(pred[SHOWCASE])
print(f"linear probe on frozen ResNet-50 features: {probe_acc[-1]:.1%} test accuracy after {EPOCHS} epochs")

The baseline: a small CNN trained **from scratch** on the very same 300 labelled photos (downscaled to 64 × 64 so it trains
quickly, with random crops and flips as data augmentation: every epoch the network sees a **new view** of each photo, as in notebook 02). Same number of epochs, every weight trained:

In [ ]:
from torchvision import transforms

class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def block(cin, cout): return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))
        self.net = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 128),
                                 nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes))
    def forward(self, x): return self.net(x)

shrink = lambda x: F.interpolate(x.float(), size=64, mode="bilinear", antialias=True)       # 224 → 64 pixels
small_train, small_test = shrink(imgs_train), shrink(imgs_test).to(device)
to_input = lambda x: (x / 255 - 0.45) / 0.25
augment = transforms.Compose([transforms.RandomResizedCrop(64, scale=(0.6, 1.0), antialias=True), transforms.RandomHorizontalFlip()])

torch.manual_seed(0)
scratch = SmallCNN(len(VESSEL_CLASSES)).to(device); opt = torch.optim.Adam(scratch.parameters(), lr=2e-3)
print(f"CNN from scratch: {sum(p.numel() for p in scratch.parameters()):,} trainable parameters")
scratch_acc, scratch_pred = [], []; t0 = time.time()
for epoch in range(EPOCHS + 1):
    if epoch > 0:
        scratch.train()
        for x, y in DataLoader(TensorDataset(small_train, labels_train), batch_size=32, shuffle=True):
            loss = F.cross_entropy(scratch(to_input(augment(x.to(device)))), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    scratch.eval()
    with torch.no_grad():
        pred = scratch(to_input(small_test)).argmax(1).cpu()
    scratch_acc.append((pred == labels_test).float().mean().item()); scratch_pred.append(pred[SHOWCASE])
print(f"small CNN from scratch: {scratch_acc[-1]:.1%} test accuracy after {EPOCHS} epochs ({time.time() - t0:.0f}s)")

In [ ]:
# @title 🎞️ Animation: the race, epoch by epoch { display-mode: "form" }
fig = plt.figure(figsize=(16, 4.6), dpi=85)
gs = fig.add_gridspec(1, 1 + len(SHOWCASE), width_ratios=[3.2] + [1] * len(SHOWCASE), wspace=0.08)
ax = fig.add_subplot(gs[0, 0])
ax.axhline(1 / len(VESSEL_CLASSES), color=GREY, ls=":", lw=1); ax.text(EPOCHS, 1 / len(VESSEL_CLASSES) + 0.015, "guessing", ha="right", color=GREY, fontsize=9)
line_p, = ax.plot([], [], "-", color=NAVY, lw=2.8, label="linear probe on frozen ResNet-50")
line_s, = ax.plot([], [], "-", color=GREY, lw=2.8, label="CNN trained from scratch")
dot_p, = ax.plot([], [], "o", color=NAVY, ms=9); dot_s, = ax.plot([], [], "o", color=GREY, ms=9)
ax.set(xlim=(0, EPOCHS), ylim=(0, 1), xlabel="epoch", ylabel="test accuracy (150 photos)"); ax.legend(loc="upper left")
cells = []
for k, q in enumerate(SHOWCASE):
    a = fig.add_subplot(gs[0, 1 + k]); a.imshow(imgs_test[q].permute(1, 2, 0)); a.axis("off")
    a.set_title(f"truth: {VESSEL_CLASSES[labels_test[q]]}", fontsize=9, color=NAVY)
    cells.append((a.text(0.5, -0.1, "", ha="center", va="top", fontsize=8.5, transform=a.transAxes),
                  a.text(0.5, -0.24, "", ha="center", va="top", fontsize=8.5, transform=a.transAxes)))
title = fig.suptitle("", color=NAVY, fontweight="bold")

def update(e):
    xs = range(e + 1)
    line_p.set_data(xs, probe_acc[:e + 1]); line_s.set_data(xs, scratch_acc[:e + 1])
    dot_p.set_data([e], [probe_acc[e]]); dot_s.set_data([e], [scratch_acc[e]])
    for k, (tp, ts) in enumerate(cells):
        truth = labels_test[SHOWCASE[k]].item()
        for txt, pred, who in [(tp, probe_pred[e][k].item(), "probe"), (ts, scratch_pred[e][k].item(), "scratch")]:
            txt.set_text(f"{who}: {VESSEL_CLASSES[pred]}"); txt.set_color(GREEN if pred == truth else RED)
    title.set_text(f"epoch {e} / {EPOCHS}:  probe {probe_acc[e]:.0%}  vs.  from scratch {scratch_acc[e]:.0%}")
    return []

play(animation.FuncAnimation(fig, update, frames=EPOCHS + 1, interval=200))

The linear probe is ahead after the first epoch and stays far ahead. With only 50 labelled photos per class, the CNN from
scratch never gets the chance to learn what a hull, a mast or a bridge looks like; the pretrained backbone learned that from
1.28 M ImageNet photos and the knowledge **transfers** to vessel types it was never trained on. This is the core promise of
foundation models.

### 🎤 Ask the audience
1. What would happen to the gap with only **5** labelled photos per class? And with **5,000**?

<details><summary><b>Answer</b></summary>

With 5 photos per class the CNN from scratch is hopeless while the probe still does reasonably well: the gap grows. With thousands of labels the scratch CNN catches up a good part of the way, at a much higher training cost.

</details>

2. What if the frozen backbone were **randomly initialised** instead of pretrained (same architecture, `weights=None`)?

<details><summary><b>Answer</b></summary>

The probe collapses: random features carry almost no information about vessel types. The architecture alone is not the magic, the pretraining is. Notebook 05 measures exactly this.

</details>

---
# Summary
1. A **Vision Foundation Model** is pretrained on a large dataset with a pretext task and reused for many downstream tasks.
   The term is **deliberately vague**: what counts is the ability to transfer, and that is a matter of degree.
   Ask *how foundational* a model is, not whether it is one.
2. **ImageNet + CNNs** (AlexNet 2012 → ResNet 2015 → EfficientNet 2019) made pretrained backbones the default.
3. A frozen pretrained backbone + a **linear probe** beats training from scratch when labels are scarce:
   with 50 photos per vessel type, roughly 78 % vs. 30–40 %.

**Next →** [04 · Vision Transformers](https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/04_vision_transformers.ipynb): replacing convolutions by attention.